In [1]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving Machines.csv to Machines.csv
Saving Maintenance.csv to Maintenance.csv
Saving production.csv to production.csv
Saving products.csv to products.csv
Saving Quality_Inspections.csv to Quality_Inspections.csv


In [3]:
print(pd.read_csv('Machines.csv'))
print(pd.read_csv('Maintenance.csv'))
print(pd.read_csv('Quality_Inspections.csv'))
print(pd.read_csv('production.csv'))
print(pd.read_csv('products.csv'))

   machine_id machine_name   factory    installation_date   shifts  \
0        M000       CNS-00  Banglore  2026-12-24 23:40:13  Morning   
1        M001       CNS-01  Karnatak  2026-07-04 14:35:29  Morning   
2        M002       CNS-02       Goa  2026-02-16 18:04:09  Morning   
3        M003       CNS-03       Goa  2026-07-03 11:20:15  Morning   
4        M004       CNS-04       Goa  2026-03-25 18:30:15  Morning   
..        ...          ...       ...                  ...      ...   
95      M0095      CNS-095    Mumbai  2026-11-26 17:58:34   Nights   
96      M0096      CNS-096  Karnatak  2026-07-20 12:21:37   Nights   
97      M0097      CNS-097    Mumbai  2026-07-30 14:40:42   Nights   
98      M0098      CNS-098       Goa  2026-09-09 21:50:27  Morning   
99      M0099      CNS-099       Goa  2026-05-08 19:57:55   Nights   

   production_line    status  
0               L2    Active  
1               L4    Active  
2               L3  Inactive  
3               L1  Inactive  
4   

In [4]:
df_production = pd.read_csv('production.csv')
df_product = pd.read_csv('products.csv')
df_machines = pd.read_csv('Machines.csv')
df_maintenance = pd.read_csv('Maintenance.csv')
df_quality = pd.read_csv('Quality_Inspections.csv')


In [5]:
for name,df in {
    'production':df_production,
    'product':df_product,
    'machines':df_machines,
    'maintenance':df_maintenance,
    'quality':df_quality
}.items():
  print(f"{name}")
  print(f"shape:{df.shape}")
  print(f"head:{df.head(10)}")
  print(f"No_of_rows:{len(df)}")
  print(f"no_of_col:{len(df.columns)}")
  print(f"duplicates:{df.duplicated().sum()}")
  print(f"sum_null_values:{df.isnull().sum()}")
  print(f"info:{df.info()}")
  print(f"describe:{df.describe()}")
  print(f"datatypes:{df.dtypes}")

production
shape:(100, 7)
head:  production_id machine_id product_id      production_date    shift  \
0        PR0000       M000       P000  2026-03-03 23:56:23  Morning   
1        PR0001       M001       P001  2026-06-09 14:45:35  Morning   
2        PR0002       M002       P002  2026-06-30 05:58:12  Morning   
3        PR0003       M003       P003  2026-07-24 22:53:32  Morning   
4        PR0004       M004       P004  2026-10-07 06:04:29  Morning   
5        PR0005       M005       P005  2026-03-14 14:22:30  Morning   
6        PR0006       M006       P006  2026-02-17 12:58:09  Morning   
7        PR0007       M007       P007  2026-12-12 23:02:30   Nights   
8        PR0008       M008       P008  2026-10-23 16:09:05  Morning   
9        PR0009       M009       P009  2026-05-17 08:02:32  Morning   

   units_produced  operating_hours  
0              13               11  
1              22                8  
2              25                2  
3              24                4  
4 

In [6]:
#Find the total units produced by each factory,
# broken down by production line,
# sorted from highest to lowest production.

df = df_production.merge(
    df_machines[['machine_id','factory','production_line']],
    on = 'machine_id',
    how = 'left'
)

result = (
    df.groupby(["factory",'production_line'], as_index = False)
    .agg(total_units_produced= ("units_produced",'sum'))
)

result = result.sort_values(
    'total_units_produced',
    ascending = False
)

print(result)


      factory production_line  total_units_produced
7         Goa              L4                   159
19     Mumbai              L4                   157
13   Karnatak              L2                   142
2    Banglore              L3                   134
1    Banglore              L2                   130
0    Banglore              L1                   119
12   Karnatak              L1                   115
15   Karnatak              L4                    95
18     Mumbai              L3                    82
14   Karnatak              L3                    79
16     Mumbai              L1                    79
6         Goa              L3                    74
9   Hyderabad              L2                    74
4         Goa              L1                    72
10  Hyderabad              L3                    54
8   Hyderabad              L1                    51
11  Hyderabad              L4                    48
17     Mumbai              L2                    33
3    Banglor

In [7]:
result = (df_production.groupby("shift",as_index= False).agg(total_units = ("units_produced","sum")))
print(result)

     shift  total_units
0  Morning          771
1   Nights          960


In [8]:
#avg operating hours per machine
result = (df_production.groupby('machine_id',as_index = False).agg(avg_hours=('operating_hours',"mean")))
result

,machine_id,avg_hours
0,M000,11.0
1,M001,8.0
2,M0010,10.0
3,M0011,6.0
4,M0012,8.0
...,...,...
95,M0095,8.0
96,M0096,10.0
97,M0097,11.0
98,M0098,12.0


In [9]:
#total maintenace cost per machine
total_maintenace_cost_per_machine = (df_maintenance.groupby("machine_id",as_index = False).agg(total_maintenace_cost=('maintenance_cost','sum')))
total_maintenace_cost_per_machine


,machine_id,total_maintenace_cost
0,M000,5418.5
1,M001,626.4
2,M0010,15845.1
3,M0011,18411.2
4,M0012,23480.1
...,...,...
95,M0095,9784.6
96,M0096,697.5
97,M0097,17758.5
98,M0098,379.3


In [10]:
#avg downtime by maintnenace type
avg_downtime_by_maintenance_type = (df_maintenance.groupby("maintenance_type",as_index= False).agg(avg_downtime=('downtime_hours','mean')))
avg_downtime_by_maintenance_type

,maintenance_type,avg_downtime
0,Corrective,7.736842
1,Predictive,2.000000
2,Preventive,2.256757


In [11]:
df_maintenance

,maintenance_id,machine_id,maintenance_date,maintenance_type,downtime_hours,maintenance_cost
0,MT0000,M000,2026-12-04 18:42:06,Predictive,2.2,5418.5
1,MT0001,M001,2026-11-04 21:28:49,Preventive,3.6,626.4
2,MT0002,M002,2026-07-05 13:34:03,Preventive,1.7,309.8
3,MT0003,M003,2026-08-25 14:58:30,Predictive,1.9,3360.1
4,MT0004,M004,2026-11-25 18:06:02,Preventive,1.3,629.9
...,...,...,...,...,...,...
95,MT00095,M0095,2026-07-29 20:38:27,Corrective,8.8,9784.6
96,MT00096,M0096,2026-11-06 23:29:55,Preventive,1.8,697.5
97,MT00097,M0097,2026-11-14 09:27:23,Corrective,4.7,17758.5
98,MT00098,M0098,2026-06-23 23:03:50,Preventive,2.0,379.3


In [12]:
#total inspected units by defect type
total_insepected_untis_by_defect_type = (df_quality.groupby('defect_type').agg(total_inspeceted_u_by_dt=("inspected_units","sum")))
total_insepected_untis_by_defect_type

,total_inspeceted_u_by_dt
defect_type,
Assembly Error,3372
Dimensional Error,4496
Surface Defect,2977


In [13]:
#total defective units by defect type
total_defective_units_by_defect_type = (df_quality.groupby('defect_type').agg(t_d_u_b_dt = ("defective_units","sum")))
total_defective_units_by_defect_type

,t_d_u_b_dt
defect_type,
Assembly Error,161
Dimensional Error,182
Surface Defect,114


In [14]:
# avg selling price per category
avg_selling_price_per_category = (df_product.groupby("category").agg(asppc = ("selling_price","mean")))
avg_selling_price_per_category

#select avg(selling_price),category
#from product
#group by category

,asppc
category,
Automotive,2618.859375
Electrical,3050.152857
Electronics,5810.702083
Industrial,4206.867917
Mechanical,1652.105455


In [15]:
#count production records per shift
count_production_record_per_shift = (df_production.groupby('shift').agg(cprps = ('production_id','count')))
count_production_record_per_shift


# select count(production_id),shift from production
# group by shift;

,cprps
shift,
Morning,47
Nights,53


In [16]:
# total production and avg operating hours per machine
total_production_and_avg_o_h_per_machine = (df_production.groupby('machine_id').agg(total_production=('units_produced','sum'),avg_operating_hour=(
    "operating_hours","mean"
)))
total_production_and_avg_o_h_per_machine

#select sum(units_produced) as total_production,avg(operating_hours) as avg_o_h
#from production
#group by machine_id

,total_production,avg_operating_hour
machine_id,,
M000,13,11.0
M001,22,8.0
M0010,21,10.0
M0011,13,6.0
M0012,13,8.0
...,...,...
M0095,14,8.0
M0096,23,10.0
M0097,22,11.0


In [17]:
# total maintenace cost and avg downtime per type
total_maintence_cost_and_avg_downtime_per_type = (df_maintenance.groupby('maintenance_type',as_index = False).agg(total_maintenace_cost = ("maintenance_cost","sum"),avg_downtme = (
    "downtime_hours","mean"
)))
total_maintence_cost_and_avg_downtime_per_type

#select sum(maintenace_cost) as total_m_c, avg(downtime) as dt
#from maintenance
# group by maintenance_type

,maintenance_type,total_maintenace_cost,avg_downtme
0,Corrective,671044.9,7.736842
1,Predictive,161551.8,2.000000
2,Preventive,19506.0,2.256757


In [18]:
#minimum and max selling price per category
min_max_selling_price_per_category = (df_product.groupby("category",as_index=False).agg(
  min_selling_price = ("selling_price",'min'),max_selling_price = ("selling_price","max")
))
min_max_selling_price_per_category


#select min(selling_price) as min_price , max(selling_price) as max_price
#from product
#group by category



,category,min_selling_price,max_selling_price
0,Automotive,993.01,4092.97
1,Electrical,1233.89,4228.74
2,Electronics,743.31,10624.62
3,Industrial,953.70,7060.98
4,Mechanical,599.20,3230.61


In [19]:
#total insepected and defective units per defect type
total_inspected_and_defective_units_per_defect_type = (
    df_quality.groupby('defect_type').agg(total_inspected_units = ('inspected_units','sum'),
                                          total_defect_units=('defective_units','sum'))
)
total_inspected_and_defective_units_per_defect_type



#select sum(inspected_units) as total_inspected_units,sum(defective_units) as total_defective_units
#from from quality_inspection
#group by defect_type

,total_inspected_units,total_defect_units
defect_type,,
Assembly Error,3372,161
Dimensional Error,4496,182
Surface Defect,2977,114


In [20]:
#five machines with the highest total production
highest_total_production_5 = (df_production.groupby('machine_id').agg(total_production = ('units_produced','sum')).sort_values('total_production',ascending=False))
highest_total_production_5


#select machine_id, sum(units_produced) as total_production
#from production
#group by machine_id
#order by total_production desc
#limit 5

,total_production
machine_id,
M002,25
M0088,25
M0066,25
M0064,25
M0077,25
...,...
M0013,10
M007,10
M0052,10


In [21]:
#product categories with avg selling price above 500
product_cat_with_avg_selling_price_gr8r_500= (df_product.groupby('category').agg(avg_selling_price = ("selling_price",'mean'))
.query('avg_selling_price > 500'))
product_cat_with_avg_selling_price_gr8r_500
#select avg(selling_price) as avg_selling_price, category
#from production
#where selling_price > 500
#group by category

,avg_selling_price
category,
Automotive,2618.859375
Electrical,3050.152857
Electronics,5810.702083
Industrial,4206.867917
Mechanical,1652.105455


In [22]:
# Total production per factory
total_production_per_factory = (df_production.merge(
    df_machines[['machine_id','factory']],
    on = 'machine_id',
    how = 'left'
).groupby('factory',as_index = False).agg(total_units = ('units_produced','sum'))
)
total_production_per_factory

#select m.machine_id,m.factory,sum(p.units_produced)
#from production as p
#left join machines as m
#on m.machine_id = p.production_id
# group by facotry

,factory,total_units
0,Banglore,404
1,Goa,318
2,Hyderabad,227
3,Karnatak,431
4,Mumbai,351


In [23]:
df_machines

,machine_id,machine_name,factory,installation_date,shifts,production_line,status
0,M000,CNS-00,Banglore,2026-12-24 23:40:13,Morning,L2,Active
1,M001,CNS-01,Karnatak,2026-07-04 14:35:29,Morning,L4,Active
2,M002,CNS-02,Goa,2026-02-16 18:04:09,Morning,L3,Inactive
3,M003,CNS-03,Goa,2026-07-03 11:20:15,Morning,L1,Inactive
4,M004,CNS-04,Goa,2026-03-25 18:30:15,Morning,L3,Active
...,...,...,...,...,...,...,...
95,M0095,CNS-095,Mumbai,2026-11-26 17:58:34,Nights,L3,Active
96,M0096,CNS-096,Karnatak,2026-07-20 12:21:37,Nights,L2,Inactive
97,M0097,CNS-097,Mumbai,2026-07-30 14:40:42,Nights,L3,Active
98,M0098,CNS-098,Goa,2026-09-09 21:50:27,Morning,L3,Inactive


In [24]:
#production per factory and production line
production_per_factory_and_line = (
    df_production
    .merge(
        df_machines[
            ["machine_id", "factory", "production_line"]
        ],
        on="machine_id",
        how="left"
    )
    .groupby(
        ["factory", "production_line"],
        as_index=False
    )
    .agg(
        total_units=("units_produced", "sum")
    )
    .sort_values("total_units", ascending=False)
)

production_per_factory_and_line

,factory,production_line,total_units
7,Goa,L4,159
19,Mumbai,L4,157
13,Karnatak,L2,142
2,Banglore,L3,134
1,Banglore,L2,130
0,Banglore,L1,119
12,Karnatak,L1,115
15,Karnatak,L4,95
18,Mumbai,L3,82
14,Karnatak,L3,79


In [25]:
# total defects per product
total_defects_per_product = (
    df_quality.merge(
        df_production[['product_id','production_id']]
    ).groupby('product_id').agg(total_defects = ('defective_units','sum'))
    .sort_values('total_defects',ascending=False)
)
total_defects_per_product

#select sum(q.defective_units) as total_defects,p.product_id
#from quality_inspection
#left join production
# on p.production_id = q.production
# group by product_id
#order by total_defects desc

,total_defects
product_id,
P0063,15
P0071,13
P0037,12
P0034,12
P0058,12
...,...
P0070,0
P0095,0
P0094,0


In [26]:
# FIVE MACHINES WITH THE HIGHEST MAINTENANCE DWONTIME
#select m.machine_id,sum(mm.downtime_hours) as highest_maintenance_dt,m.machine_name
# from maintenece as mm
#left join machines as m
# on m.machine_id = mm.machine_id
# group by machine_name,machie_id
#order by highest_maintence_downtime
#limit 5

highest_maintence_downtime_by_machines = (df_maintenance.merge(
    df_machines[['machine_id','machine_name','factory']],on = 'machine_id',how = 'left')
    .groupby(['machine_id','machine_name','factory']).agg(highest_maintenace_dt=('downtime_hours','sum'))
    .sort_values('highest_maintenace_dt',ascending = False)
)

highest_maintence_downtime_by_machines


,,,highest_maintenace_dt
machine_id,machine_name,factory,
M0075,CNS-075,Mumbai,12.0
M0012,CNS-012,Goa,11.5
M0016,CNS-016,Banglore,11.3
M0052,CNS-052,Hyderabad,11.2
M0059,CNS-059,Mumbai,10.8
...,...,...,...
M0017,CNS-017,Banglore,1.1
M0093,CNS-093,Mumbai,1.1
M0099,CNS-099,Goa,1.1


In [27]:
datasets = {
    "production": df_production,
    "products": df_product,
    "machines": df_machines,
    "maintenance": df_maintenance,
    "quality": df_quality
}

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print("Rows and columns:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values per column:")
    print(df.isnull().sum())



--- PRODUCTION ---
Rows and columns: (100, 7)
Duplicate rows: 0
Missing values per column:
production_id      0
machine_id         0
product_id         0
production_date    0
shift              0
units_produced     0
operating_hours    0
dtype: int64

--- PRODUCTS ---
Rows and columns: (100, 6)
Duplicate rows: 0
Missing values per column:
product_id                  0
product_name                0
category                    0
unit_cost                   0
selling_price               0
standard_production_time    0
dtype: int64

--- MACHINES ---
Rows and columns: (100, 7)
Duplicate rows: 0
Missing values per column:
machine_id           0
machine_name         0
factory              0
installation_date    0
shifts               0
production_line      0
status               0
dtype: int64

--- MAINTENANCE ---
Rows and columns: (100, 6)
Duplicate rows: 0
Missing values per column:
maintenance_id      0
machine_id          0
maintenance_date    0
maintenance_type    0
downtime_hours      

In [28]:
primary_keys = {
    "production": (df_production, "production_id"),
    "products": (df_product, "product_id"),
    "machines": (df_machines, "machine_id"),
    "maintenance": (df_maintenance, "maintenance_id"),
    "quality": (df_quality, "inspection_id")
}

for name, (df, key) in primary_keys.items():
    duplicate_count = df[key].duplicated().sum()
    null_count = df[key].isnull().sum()

    print(f"\n--- {name.upper()} ---")
    print("Primary key:", key)
    print("Duplicate keys:", duplicate_count)
    print("Null keys:", null_count)
    print("Status:", "PASS" if duplicate_count == 0 and null_count == 0 else "FAIL")


--- PRODUCTION ---
Primary key: production_id
Duplicate keys: 0
Null keys: 0
Status: PASS

--- PRODUCTS ---
Primary key: product_id
Duplicate keys: 0
Null keys: 0
Status: PASS

--- MACHINES ---
Primary key: machine_id
Duplicate keys: 0
Null keys: 0
Status: PASS

--- MAINTENANCE ---
Primary key: maintenance_id
Duplicate keys: 0
Null keys: 0
Status: PASS

--- QUALITY ---
Primary key: inspection_id
Duplicate keys: 0
Null keys: 0
Status: PASS


In [29]:
# Check whether every foreign key matches a parent record

checks = [
    (
        "Production -> Machines",
        df_production,
        "machine_id",
        df_machines,
        "machine_id"
    ),
    (
        "Production -> Products",
        df_production,
        "product_id",
        df_product,
        "product_id"
    ),
    (
        "Maintenance -> Machines",
        df_maintenance,
        "machine_id",
        df_machines,
        "machine_id"
    ),
    (
        "Quality -> Production",
        df_quality,
        "production_id",
        df_production,
        "production_id"
    )
]

for name, child, fk, parent, pk in checks:
    unmatched = child.loc[
        ~child[fk].isin(parent[pk]),
        fk
    ]

    print(f"\n--- {name} ---")
    print("Unmatched foreign keys:", len(unmatched))
    print("Status:", "PASS" if unmatched.empty else "FAIL")

    if not unmatched.empty:
        print(unmatched.unique())


--- Production -> Machines ---
Unmatched foreign keys: 0
Status: PASS

--- Production -> Products ---
Unmatched foreign keys: 0
Status: PASS

--- Maintenance -> Machines ---
Unmatched foreign keys: 0
Status: PASS

--- Quality -> Production ---
Unmatched foreign keys: 0
Status: PASS


In [30]:
(
    "Production -> Machines",
    df_production,
    "machine_id",
    df_machines,
    "machine_id"
)

('Production -> Machines',
    production_id machine_id product_id      production_date    shift  \
 0         PR0000       M000       P000  2026-03-03 23:56:23  Morning   
 1         PR0001       M001       P001  2026-06-09 14:45:35  Morning   
 2         PR0002       M002       P002  2026-06-30 05:58:12  Morning   
 3         PR0003       M003       P003  2026-07-24 22:53:32  Morning   
 4         PR0004       M004       P004  2026-10-07 06:04:29  Morning   
 ..           ...        ...        ...                  ...      ...   
 95       PR00095      M0095      P0095  2026-12-14 19:06:10   Nights   
 96       PR00096      M0096      P0096  2026-03-19 12:43:34   Nights   
 97       PR00097      M0097      P0097  2026-11-21 19:42:24   Nights   
 98       PR00098      M0098      P0098  2026-05-22 11:42:28  Morning   
 99       PR00099      M0099      P0099  2026-10-10 11:58:28   Nights   
 
     units_produced  operating_hours  
 0               13               11  
 1               

In [31]:
checks = [
    ("Production -> Machines", df_production, "machine_id", df_machines, "machine_id"),
    ("Production -> Products", df_production, "product_id", df_product, "product_id"),
    ("Maintenance -> Machines", df_maintenance, "machine_id", df_machines, "machine_id"),
    ("Quality -> Production", df_quality, "production_id", df_production, "production_id")
]

for name, child, fk, parent, pk in checks:
    missing = (~child[fk].isin(parent[pk])).sum()
    print(f"{name}: {'PASS' if missing == 0 else 'FAIL'} — {missing} unmatched")

Production -> Machines: PASS — 0 unmatched
Production -> Products: PASS — 0 unmatched
Maintenance -> Machines: PASS — 0 unmatched
Quality -> Production: PASS — 0 unmatched
